# AI Media Studio — Google Colab Pro L4 Worker v4.0 (Clean-Slate · Zero Drive)

Arquitetura à prova de falhas para **NVIDIA L4 (24 GB VRAM + RAM Alta)** em NVMe local (`/content/ai_studio_v4`), 100% sem Google Drive:
- **Isolamento hermético (`clean_env.py`):** neutraliza `MPLBACKEND=module://matplotlib_inline...`, `PYTHONPATH`, `PYTHONHOME`, `JPY_*`, `COLAB_*` (preservando `COLAB_AUTH_TOKEN`) e `LD_PRELOAD` em todas as células e no bootstrap do servidor, preservando e enriquecendo `LD_LIBRARY_PATH` (`/usr/lib64-nvidia`, `/usr/local/cuda/lib64`, `/usr/local/cuda-12*/targets/x86_64-linux/lib`), `CUDA_*` e `NVIDIA_*`.
- **Dual Venv Python 3.10 via `uv` imune a `libnppicc.so.12`:** `envs/seedvc` fixo em `torch==2.7.1+cu126` + `torchaudio==2.7.1+cu126` (nativo `soundfile`, zero `torchcodec`); `envs/wangp` em `torch==2.10.0+cu128` + `torchcodec==0.10.0` da wheel CPU (`--no-deps --index-url .../whl/cpu`) com fallback automático verificado para `cu126` (`2.7.1`) se necessário.
- **Configuração por família validada no upstream `WanGP`:** `vae_block_size=64` + `audio_prompt_type='AS'` para LTX-2.5, `image_start` (`S`) + `image_refs` (`KI`) + `distill` para LongCat Avatar 1.6/1.5, `image_start` + `audio_prompt_type='A'` para InfiniteTalk, com `INT8` (Transformer + Text Encoder) + `SDPA` + `Profile 4`.
- **Smoke test real + Quality Gates + Watchdog HTTP/2:** Célula 2 valida `torchaudio.save/load` + CUDA nos dois venvs; Célula 3 valida conversão real Seed-VC (Audio Quality Gate PCM) e micro-render real WanGP (Video Quality Gate `ffprobe` + liberação de VRAM); Célula 4 expõe `version='4.0.3-clean-l4'` (compatível com `vps_novela_runner.py`), valida `GET <tunnel>/health` externo e auto-recupera túnel/servidor sem caixas vermelhas.


In [ ]:
# @title 1. Limpeza Total, Sanitizador Hermético & Pré-Requisitos (Zero Google Drive)
REBUILD_FROM_SCRATCH = True  # @param {type:'boolean'}

import json, os, re, shutil, subprocess, sys, time, warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Bracket trick evita que pkill mate o próprio subshell bash -lc
subprocess.run(
    ["bash", "-lc", "pkill -f '[c]olab_v4_cinema_server' || true; pkill -f '[c]olab_gpu_server' || true; pkill -f '[c]loudflared' || true"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(1)
subprocess.run(
    ["bash", "-lc", "pkill -9 -f '[c]olab_v4_cinema_server' || true; pkill -9 -f '[c]loudflared' || true"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

legacy_dirs = [
    "/content/ai_studio_cache", "/content/venv_cinema311", "/content/seed-vc",
    "/content/WanGP", "/content/LivePortrait", "/content/LatentSync", "/content/MuseTalk",
]
if REBUILD_FROM_SCRATCH:
    legacy_dirs.append("/content/ai_studio_v4")
for ld in legacy_dirs:
    shutil.rmtree(ld, ignore_errors=True)

WS = Path("/content/ai_studio_v4")
for sub in (
    "repos", "envs", "hf_cache/hub", "models/seedvc", "models/wangp",
    "config", "outputs/wangp", "tmp",
    "cache/triton", "cache/inductor", "cache/matplotlib", "cache/xdg",
):
    (WS / sub).mkdir(parents=True, exist_ok=True)

CLEAN_ENV_CODE = '"""Deterministic environment sanitizer for Colab Pro L4 Worker v4.0."""\nfrom __future__ import annotations\n\nimport os\nfrom pathlib import Path\nfrom typing import Dict, Optional\n\nTOXIC_EXACT_VARS = {\n    "MPLBACKEND",\n    "PYTHONHOME",\n    "PYTHONPATH",\n    "PYTHONSTARTUP",\n    "PYTHONUSERBASE",\n    "PYTHONEXECUTABLE",\n    "CONDA_PREFIX",\n    "CONDA_DEFAULT_ENV",\n    "CONDA_SHLVL",\n    "PIP_CONSTRAINT",\n    "PIP_REQUIRE_VIRTUALENV",\n    "UV_PYTHON",\n    "JPY_PARENT_PID",\n    "JPY_SESSION_NAME",\n    "JPY_API_TOKEN",\n    "IPYTHONDIR",\n    "DISPLAY",\n    "LD_PRELOAD",\n}\n\nTOXIC_PREFIXES = (\n    "JPY_",\n    "JUPYTER_",\n    "IPYTHON_",\n    "COLAB_",\n    "TBE_",\n)\n\nPRESERVED_EXACT_VARS = {\n    "COLAB_AUTH_TOKEN",\n}\n\n\ndef _augment_cuda_ld_library_path(existing: str) -> str:\n    """Append existing host CUDA/NVIDIA library dirs without clobbering existing entries."""\n    candidates: list[str] = []\n    for base in ("/usr/lib64-nvidia", "/usr/local/cuda/lib64"):\n        if Path(base).is_dir():\n            candidates.append(base)\n    usr_local = Path("/usr/local")\n    if usr_local.is_dir():\n        for p in sorted(usr_local.glob("cuda-12*/targets/x86_64-linux/lib")):\n            if p.is_dir():\n                candidates.append(str(p))\n    parts = [x for x in (existing or "").split(":") if x]\n    for c in candidates:\n        if c not in parts:\n            parts.append(c)\n    return ":".join(parts)\n\n\ndef build_clean_env(\n    ws: Path = Path("/content/ai_studio_v4"),\n    venv_dir: Optional[Path] = None,\n    extra: Optional[Dict[str, str]] = None,\n    base_env: Optional[Dict[str, str]] = None,\n) -> Dict[str, str]:\n    """Return a clean subprocess env preserving CUDA/NVIDIA libs & COLAB_AUTH_TOKEN."""\n    src = dict(os.environ if base_env is None else base_env)\n    cleaned: Dict[str, str] = {}\n    for k, v in src.items():\n        if k in PRESERVED_EXACT_VARS:\n            cleaned[k] = v\n            continue\n        if k in TOXIC_EXACT_VARS or k == "VIRTUAL_ENV":\n            continue\n        if any(k.startswith(pref) for pref in TOXIC_PREFIXES):\n            continue\n        cleaned[k] = v\n\n    ld_path = _augment_cuda_ld_library_path(cleaned.get("LD_LIBRARY_PATH", ""))\n    if ld_path:\n        cleaned["LD_LIBRARY_PATH"] = ld_path\n\n    cache_dir = ws / "cache"\n    hf_cache = ws / "hf_cache"\n    cleaned.update({\n        "MPLBACKEND": "Agg",\n        "QT_QPA_PLATFORM": "offscreen",\n        "PYGAME_HIDE_SUPPORT_PROMPT": "1",\n        "TOKENIZERS_PARALLELISM": "false",\n        "PIP_NO_INPUT": "1",\n        "PIP_NO_WARN_SCRIPT_LOCATION": "1",\n        "UV_NO_PROGRESS": "1",\n        "UV_LINK_MODE": "copy",\n        "PYTHONWARNINGS": "ignore",\n        "PYTHONUNBUFFERED": "1",\n        "PYTHONNOUSERSITE": "1",\n        "DEBIAN_FRONTEND": "noninteractive",\n        "HF_HUB_DISABLE_PROGRESS_BARS": "1",\n        "HF_HUB_VERBOSITY": "error",\n        "HF_HOME": str(hf_cache),\n        "HF_HUB_CACHE": str(hf_cache / "hub"),\n        "TRITON_CACHE_DIR": str(cache_dir / "triton"),\n        "TORCHINDUCTOR_CACHE_DIR": str(cache_dir / "inductor"),\n        "MPLCONFIGDIR": str(cache_dir / "matplotlib"),\n        "XDG_CACHE_HOME": str(cache_dir / "xdg"),\n        "WAN2GP_DISABLE_AUDIO": "1",\n        "TF_CPP_MIN_LOG_LEVEL": "3",\n        "LC_ALL": "C.UTF-8",\n        "LANG": "C.UTF-8",\n    })\n\n    if venv_dir is not None:\n        vbin = str(Path(venv_dir) / "bin")\n        cleaned["VIRTUAL_ENV"] = str(venv_dir)\n        old_path = cleaned.get("PATH", "/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin")\n        parts = [p for p in old_path.split(":") if p and p != vbin]\n        cleaned["PATH"] = ":".join([vbin] + parts)\n\n    if extra:\n        for ek, ev in extra.items():\n            if ev is not None:\n                cleaned[ek] = str(ev)\n\n    return cleaned\n'
(WS / "clean_env.py").write_text(CLEAN_ENV_CODE, encoding="utf-8")
if str(WS) not in sys.path:
    sys.path.insert(0, str(WS))
from clean_env import build_clean_env  # noqa: E402

SYS_ENV = build_clean_env(ws=WS)
assert SYS_ENV["MPLBACKEND"] == "Agg"
assert "PYTHONPATH" not in SYS_ENV and "JPY_PARENT_PID" not in SYS_ENV
print(f"[OK] Workspace limpo + sanitizador clean_env.py prontos em {WS}")

def run_clean(label, cmd, cwd=None, env=None):
    t0 = time.time()
    p = subprocess.run(
        cmd,
        cwd=cwd,
        env=env or SYS_ENV,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    dt = time.time() - t0
    if p.returncode != 0:
        tail = "\n".join((p.stdout or "").splitlines()[-60:])
        raise RuntimeError(f"[ERRO] {label} falhou (rc={p.returncode}):\n{tail}")
    print(f"[OK] {label} ({dt:.1f}s)")
    return (p.stdout or "").strip()

smi_gpu = run_clean("Detecção GPU NVIDIA", [
    "nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"
])
if not smi_gpu or "NVIDIA" not in smi_gpu.upper():
    raise RuntimeError("GPU NVIDIA não detectada! Altere o Runtime do Colab para GPU L4.")

smi_full = subprocess.check_output(["nvidia-smi"], env=SYS_ENV, text=True)
m_cuda = re.search(r"CUDA Version:\s*(\d+)\.(\d+)", smi_full)
cuda_major, cuda_minor = (int(m_cuda.group(1)), int(m_cuda.group(2))) if m_cuda else (12, 6)

if (cuda_major, cuda_minor) >= (12, 8):
    cuda_cfg = {"idx": "cu128", "torch": "2.10.0", "tv": "0.25.0", "ta": "2.10.0", "torchcodec": "0.10.0"}
elif (cuda_major, cuda_minor) >= (12, 6):
    cuda_cfg = {"idx": "cu126", "torch": "2.7.1", "tv": "0.22.1", "ta": "2.7.1", "torchcodec": ""}
else:
    cuda_cfg = {"idx": "cu124", "torch": "2.6.0", "tv": "0.21.0", "ta": "2.6.0", "torchcodec": ""}
cuda_cfg["driver_cuda"] = f"{cuda_major}.{cuda_minor}"
(WS / "config/cuda_target.json").write_text(json.dumps(cuda_cfg, indent=2), encoding="utf-8")

disk_free_gb = round(shutil.disk_usage("/content").free / (1024 ** 3), 1)
print(
    f"[INFO] GPU: {smi_gpu} | Driver CUDA: {cuda_major}.{cuda_minor} | "
    f"PyTorch Alvo: {cuda_cfg['torch']}+{cuda_cfg['idx']} | Disco /content livre: {disk_free_gb} GB"
)

run_clean("apt-get dependências de mídia, OpenCV e build", [
    "bash", "-lc",
    "apt-get update -qq && apt-get install -y -qq "
    "ffmpeg libsndfile1 libportaudio2 libgl1 libglib2.0-0 libsm6 libxext6 libxrender1 "
    "build-essential git curl pkg-config"
])
run_clean("Instalação uv (gerenciador rápido em Rust)", [
    sys.executable, "-m", "pip", "install", "-q", "--no-input", "--upgrade", "uv"
])
run_clean("Download cloudflared CLI", [
    "bash", "-lc",
    "curl -fsSL -o /usr/local/bin/cloudflared "
    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
    "&& chmod +x /usr/local/bin/cloudflared"
])
cf_ver = run_clean("Verificação cloudflared", ["cloudflared", "--version"])
ff_ver = run_clean("Verificação ffmpeg", ["bash", "-lc", "ffmpeg -version | head -n 1"])
uv_ver = run_clean("Verificação uv", ["uv", "--version"])
print(f"[OK] Sistema base pronto | {ff_ver} | {uv_ver} | {cf_ver.splitlines()[0]}")


In [ ]:
# @title 2. Provisionamento Dual-Venv Python 3.10 (Seed-VC + WanGP · Isolamento Hermético)
VIDEO_MODEL = 'ltx2_25_22B_distilled'  # @param ['ltx2_25_22B_distilled', 'longcat_avatar_v1_6', 'longcat_avatar_v1_5', 'infinitetalk', 'i2v_2_2', 'seed_vc_only']
PRELOAD_WEIGHTS = True  # @param {type:'boolean'}
HF_TOKEN = ''  # @param {type:'string'}
COLAB_AUTH_TOKEN = ''  # @param {type:'string'}

import json, subprocess, sys, time, warnings
from pathlib import Path

warnings.filterwarnings("ignore")

WS = Path("/content/ai_studio_v4")
if str(WS) not in sys.path:
    sys.path.insert(0, str(WS))
from clean_env import build_clean_env  # noqa: E402

cuda_cfg = json.loads((WS / "config/cuda_target.json").read_text(encoding="utf-8"))

def _read_colab_secret(name: str) -> str:
    try:
        from google.colab import userdata
        return (userdata.get(name) or "").strip()
    except Exception:
        return ""

hf_tok = (HF_TOKEN or "").strip() or _read_colab_secret("HF_TOKEN")
auth_tok = (COLAB_AUTH_TOKEN or "").strip() or _read_colab_secret("COLAB_AUTH_TOKEN")

worker_env_cfg = {
    "VIDEO_MODEL": VIDEO_MODEL,
    "PRELOAD_WEIGHTS": bool(PRELOAD_WEIGHTS),
    "HF_TOKEN": hf_tok,
    "COLAB_AUTH_TOKEN": auth_tok,
}
(WS / "config/worker_env.json").write_text(json.dumps(worker_env_cfg, indent=2), encoding="utf-8")

extra_hf = {"HF_TOKEN": hf_tok} if hf_tok else None
BASE_ENV = build_clean_env(ws=WS, extra=extra_hf)

def run_clean(label, cmd, cwd=None, venv_dir=None, extra_env=None):
    t0 = time.time()
    merged_extra = dict(extra_hf or {})
    if extra_env:
        merged_extra.update(extra_env)
    env = build_clean_env(ws=WS, venv_dir=venv_dir, extra=merged_extra or None) if venv_dir else BASE_ENV
    p = subprocess.run(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    dt = time.time() - t0
    if p.returncode != 0:
        tail = "\n".join((p.stdout or "").splitlines()[-65:])
        raise RuntimeError(f"[ERRO] {label} falhou (rc={p.returncode}):\n{tail}")
    print(f"[OK] {label} ({dt:.1f}s)")
    return (p.stdout or "").strip()

run_clean("Provisionamento Python 3.10 (uv)", ["uv", "python", "install", "3.10"])

TORCH_VER = cuda_cfg["torch"]
TV_VER = cuda_cfg["tv"]
TA_VER = cuda_cfg["ta"]
TC_VER = cuda_cfg.get("torchcodec") or ""
TORCH_INDEX = f"https://download.pytorch.org/whl/{cuda_cfg['idx']}"

# --- 1. Ambiente Seed-VC (100% imune a torchcodec/libnppicc.so.12: PyTorch 2.7.1+cu126 + soundfile nativo) ---
SEEDVC_VENV = WS / "envs/seedvc"
SEEDVC_PY = str(SEEDVC_VENV / "bin/python")
SEEDVC_DIR = WS / "repos/seed-vc"

if not (SEEDVC_DIR / "inference.py").is_file():
    run_clean("Clone Plachtaa/seed-vc", [
        "git", "clone", "--depth", "1", "https://github.com/Plachtaa/seed-vc.git", str(SEEDVC_DIR)
    ])

ckpt_link = SEEDVC_DIR / "checkpoints"
if not ckpt_link.exists() and not ckpt_link.is_symlink():
    ckpt_link.symlink_to(WS / "models/seedvc")

run_clean("Criação venv Python 3.10 Seed-VC", ["uv", "venv", str(SEEDVC_VENV), "--python", "3.10", "--seed"])

# L4 (Ada Lovelace sm_89) com driver CUDA 12.8+/13.0 executa cu126 nativamente;
# torchaudio==2.7.1 usa soundfile direto sem tocar em libtorchcodec_core*.so nem libnppicc.so.12.
run_clean("Instalação PyTorch 2.7.1+cu126 + torchaudio 2.7.1 (Seed-VC · sem torchcodec)", [
    "uv", "pip", "install", "--python", SEEDVC_PY,
    "--index-url", "https://download.pytorch.org/whl/cu126",
    "torch==2.7.1", "torchaudio==2.7.1",
], venv_dir=SEEDVC_VENV)

svc_pkgs = [
    "numpy==1.26.4", "transformers==4.46.3", "accelerate",
    "huggingface-hub>=0.28.1", "safetensors", "tqdm",
    "scipy==1.13.1", "librosa==0.10.2", "munch==4.0.0", "einops==0.8.0",
    "descript-audio-codec==1.0.0", "soundfile==0.12.1", "pydub==0.25.1",
    "pyyaml", "resampy", "hydra-core==1.3.2", "python-dotenv", "jiwer==3.0.3",
]
run_clean("Instalação dependências limpas Seed-VC (soundfile nativo, zero GUI/pyaudio/torchcodec)", [
    "uv", "pip", "install", "--python", SEEDVC_PY, *svc_pkgs,
], venv_dir=SEEDVC_VENV)

# Valida imports + CUDA + torchaudio.save() + torchaudio.load() reais sem torchcodec
svc_check_code = (
    "import os, torch, torchaudio, transformers, librosa, munch, soundfile, dac\n"
    "from pathlib import Path\n"
    "assert os.environ.get('MPLBACKEND') == 'Agg'\n"
    "assert 'PYTHONPATH' not in os.environ\n"
    "assert torch.cuda.is_available()\n"
    "chk = Path('/content/ai_studio_v4/tmp/ta_save_check.wav')\n"
    "torchaudio.save(str(chk), torch.zeros(1, 1600), 16000)\n"
    "wav, sr = torchaudio.load(str(chk))\n"
    "assert chk.is_file() and chk.stat().st_size > 100 and sr == 16000 and tuple(wav.shape) == (1, 1600)\n"
    "chk.unlink(missing_ok=True)\n"
    "print('SEEDVC_ENV_OK', torch.__version__, torchaudio.__version__, torch.cuda.get_device_name(0))\n"
)
svc_check_py = WS / "tmp/seedvc_selfcheck.py"
svc_check_py.write_text(svc_check_code, encoding="utf-8")
svc_check = run_clean("Self-check Seed-VC (imports + CUDA + torchaudio.save/load)", [
    SEEDVC_PY, str(svc_check_py),
], venv_dir=SEEDVC_VENV)
print("   ->", svc_check.splitlines()[-1])

# --- 2. Ambiente Worker/WanGP (cu128 + torchcodec CPU --no-deps c/ fallback verificado p/ cu126) ---
WANGP_VENV = WS / "envs/wangp"
WANGP_PY = str(WANGP_VENV / "bin/python")
WANGP_DIR = WS / "repos/WanGP"

run_clean("Criação venv Python 3.10 Worker/WanGP", ["uv", "venv", str(WANGP_VENV), "--python", "3.10", "--seed"])

run_clean(f"Instalação PyTorch {TORCH_VER}+{cuda_cfg['idx']} (Worker/WanGP)", [
    "uv", "pip", "install", "--python", WANGP_PY,
    "--index-url", TORCH_INDEX,
    f"torch=={TORCH_VER}", f"torchvision=={TV_VER}", f"torchaudio=={TA_VER}",
], venv_dir=WANGP_VENV)

if cuda_cfg["idx"] == "cu128" and TC_VER:
    # Wheel CPU do torchcodec usa apenas ffmpeg do sistema e NÃO linka contra libnppicc.so.12
    run_clean(f"Instalação torchcodec=={TC_VER} (wheel CPU --no-deps, sem libnppicc.so.12)", [
        "uv", "pip", "install", "--python", WANGP_PY,
        "--no-deps", "--index-url", "https://download.pytorch.org/whl/cpu",
        f"torchcodec=={TC_VER}",
    ], venv_dir=WANGP_VENV)

if VIDEO_MODEL == "seed_vc_only":
    run_clean("Instalação FastAPI + Uvicorn (seed_vc_only)", [
        "uv", "pip", "install", "--python", WANGP_PY,
        "fastapi", "uvicorn[standard]", "pydantic>=2.9", "python-multipart",
    ], venv_dir=WANGP_VENV)
else:
    if not (WANGP_DIR / "wgp.py").is_file():
        run_clean("Clone deepbeepmeep/WanGP", [
            "git", "clone", "--depth", "1", "https://github.com/deepbeepmeep/WanGP.git", str(WANGP_DIR)
        ])
    ckpts_link = WANGP_DIR / "ckpts"
    if not ckpts_link.exists() and not ckpts_link.is_symlink():
        ckpts_link.symlink_to(WS / "models/wangp")

    kept_reqs = ["fastapi", "uvicorn[standard]", "python-multipart", "numpy==2.1.2", "cython", "setuptools", "wheel"]
    # Exclui torch/torchvision/torchaudio/torchcodec/numpy para não sobrescrever os wheels CUDA/CPU dedicados
    skip_prefixes = ("torch", "torchvision", "torchaudio", "torchcodec", "numpy")
    for raw in (WANGP_DIR / "requirements.txt").read_text(encoding="utf-8").splitlines():
        ln = raw.split("#", 1)[0].strip()
        if not ln:
            continue
        low = ln.lower()
        if low.startswith("--extra-index-url"):
            continue
        if "onnxruntime-gpu" in low and "3.11" in low:
            continue
        if low in skip_prefixes or any(
            low.startswith(p + "=") or low.startswith(p + ">") or low.startswith(p + "<") or low.startswith(p + " ")
            for p in skip_prefixes
        ):
            continue
        if low.startswith("insightface"):
            continue
        kept_reqs.append(ln)

    req_file = WS / "tmp/wangp_reqs_clean.txt"
    req_file.write_text("\n".join(kept_reqs) + "\n", encoding="utf-8")
    run_clean(f"Instalação dependências oficiais WanGP + FastAPI ({len(kept_reqs)} pacotes)", [
        "uv", "pip", "install", "--python", WANGP_PY, "-r", str(req_file),
    ], venv_dir=WANGP_VENV)
    run_clean("Compilação insightface==0.7.3 (Linux Python 3.10)", [
        "uv", "pip", "install", "--python", WANGP_PY, "--no-build-isolation", "insightface==0.7.3",
    ], venv_dir=WANGP_VENV)

    wgp_selfcheck = (
        "import os, sys, matplotlib, torch, torchaudio, fastapi, uvicorn\n"
        "from pathlib import Path\n"
        "assert os.environ.get('MPLBACKEND') == 'Agg'\n"
        "assert matplotlib.get_backend().lower() == 'agg'\n"
        "assert 'PYTHONPATH' not in os.environ\n"
        "assert torch.cuda.is_available()\n"
        "chk = Path('/content/ai_studio_v4/tmp/ta_wgp_check.wav')\n"
        "torchaudio.save(str(chk), torch.zeros(1, 1600), 16000)\n"
        "wav, sr = torchaudio.load(str(chk))\n"
        "assert chk.is_file() and chk.stat().st_size > 100 and sr == 16000 and tuple(wav.shape) == (1, 1600)\n"
        "chk.unlink(missing_ok=True)\n"
        "sys.path.insert(0, '/content/ai_studio_v4/repos/WanGP')\n"
        "from shared.utils import generation_overrides\n"
        "ov = generation_overrides.parse_overrides(\n"
        "    \"attention_mode='sdpa', memory_profile=4, transformer_quantization='int8', text_encoder_quantization='int8'\"\n"
        ")\n"
        "assert ov['memory_profile'] == 4 and ov['transformer_quantization'] == 'int8'\n"
        "from shared.api import init\n"
        "s = init(\n"
        "    root=Path('/content/ai_studio_v4/repos/WanGP'),\n"
        "    config_path=Path('/content/ai_studio_v4/config/wgp_config.json'),\n"
        "    output_dir=Path('/content/ai_studio_v4/outputs/wangp'),\n"
        "    cli_args=['--attention', 'sdpa', '--profile', '4'],\n"
        "    console_output=False,\n"
        ")\n"
        f"target = {VIDEO_MODEL!r}\n"
        "all_models = s.list_model_metadata() or []\n"
        "m = s.get_model_metadata(target)\n"
        "if not m:\n"
        "    ids = [x.get('id') or x.get('model_type') or str(x) for x in all_models]\n"
        "    raise RuntimeError(f'Modelo {target!r} ausente no WanGP clonado. IDs disponíveis: {ids[:40]}')\n"
        "print('WANGP_API_OK', torch.__version__, torchaudio.__version__, 'models=', len(all_models), 'target=', target)\n"
    )
    sc_path = WS / "tmp/wangp_selfcheck.py"
    sc_path.write_text(wgp_selfcheck, encoding="utf-8")
    try:
        wgp_check = run_clean("Self-check WanGP (torchaudio.save/load + shared.api.init + overrides + Agg + CUDA)", [
            WANGP_PY, str(sc_path),
        ], venv_dir=WANGP_VENV)
    except RuntimeError as exc:
        msg = str(exc)
        if cuda_cfg["idx"] == "cu128" and any(tok in msg for tok in ("libtorchcodec", "torchcodec", "undefined symbol", "torchaudio")):
            print("[WARN] torchcodec CPU apresentou incompatibilidade com torch 2.10+cu128 — aplicando fallback verificado para PyTorch 2.7.1+cu126 (sem torchcodec)...")
            run_clean("Remoção torchcodec (fallback cu126)", [
                "uv", "pip", "uninstall", "--python", WANGP_PY, "torchcodec",
            ], venv_dir=WANGP_VENV)
            run_clean("Instalação PyTorch 2.7.1+cu126 + torchvision 0.22.1 + torchaudio 2.7.1 (WanGP fallback)", [
                "uv", "pip", "install", "--python", WANGP_PY,
                "--index-url", "https://download.pytorch.org/whl/cu126",
                "torch==2.7.1", "torchvision==0.22.1", "torchaudio==2.7.1",
            ], venv_dir=WANGP_VENV)
            wgp_check = run_clean("Self-check WanGP (fallback cu126: torchaudio.save/load + shared.api.init + CUDA)", [
                WANGP_PY, str(sc_path),
            ], venv_dir=WANGP_VENV)
        else:
            raise
    print("   ->", wgp_check.splitlines()[-1])

print("Etapa 2 concluída: 0 conflitos, sanitização verificada e 100% dos imports OK na L4.")


In [ ]:
# @title 3. Pré-Download de Pesos & Smoke Test Real End-to-End na L4 GPU
import array, json, math, shutil, struct, subprocess, sys, time, warnings, wave
from pathlib import Path

warnings.filterwarnings("ignore")

WS = Path("/content/ai_studio_v4")
if str(WS) not in sys.path:
    sys.path.insert(0, str(WS))
from clean_env import build_clean_env  # noqa: E402

env_cfg = json.loads((WS / "config/worker_env.json").read_text(encoding="utf-8"))
VIDEO_MODEL = env_cfg["VIDEO_MODEL"]
PRELOAD_WEIGHTS = env_cfg["PRELOAD_WEIGHTS"]
extra_hf = {"HF_TOKEN": env_cfg["HF_TOKEN"]} if env_cfg.get("HF_TOKEN") else None

SEEDVC_VENV = WS / "envs/seedvc"
SEEDVC_PY = str(SEEDVC_VENV / "bin/python")
WANGP_VENV = WS / "envs/wangp"
WANGP_PY = str(WANGP_VENV / "bin/python")

# TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 apenas para Seed-VC (checkpoints legados PyTorch 2.4 em PyTorch 2.10)
svc_extra = dict(extra_hf or {})
svc_extra["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
SVC_ENV = build_clean_env(ws=WS, venv_dir=SEEDVC_VENV, extra=svc_extra)
WGP_ENV = build_clean_env(ws=WS, venv_dir=WANGP_VENV, extra=extra_hf)

def write_test_wav(path: Path, freq: float = 220.0, dur: float = 1.4, sr: int = 16000) -> None:
    n = int(sr * dur)
    samples = bytearray()
    for i in range(n):
        t = i / sr
        env = math.sin(math.pi * min(1.0, i / 800.0))
        val = int(12000 * env * (0.65 * math.sin(2 * math.pi * freq * t) + 0.35 * math.sin(2 * math.pi * freq * 2.01 * t)))
        samples.extend(struct.pack("<h", val))
    with wave.open(str(path), "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sr)
        wf.writeframes(bytes(samples))

def verify_wav_not_silent(wav_path: Path) -> tuple[float, float]:
    norm = wav_path.with_suffix(".pcm16.wav")
    subprocess.run(
        ["ffmpeg", "-y", "-i", str(wav_path), "-ar", "16000", "-ac", "1", "-c:a", "pcm_s16le", str(norm)],
        env=SVC_ENV, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True,
    )
    with wave.open(str(norm), "rb") as wf:
        dur = wf.getnframes() / float(wf.getframerate())
        raw = wf.readframes(wf.getnframes())
    arr = array.array("h")
    arr.frombytes(raw)
    peak = max((abs(x) for x in arr), default=0)
    rms = math.sqrt(sum((x / 32768.0) ** 2 for x in arr) / max(1, len(arr)))
    rms_dbfs = 20.0 * math.log10(max(rms, 1e-12))
    if dur < 0.3 or rms_dbfs < -55.0 or peak < 64:
        raise RuntimeError(f"Audio Quality Gate falhou no smoke test: dur={dur:.2f}s rms={rms_dbfs:.1f}dBFS peak={peak}")
    return round(dur, 2), round(rms_dbfs, 1)

def verify_mp4_valid(mp4_path: Path) -> dict:
    if not mp4_path.is_file() or mp4_path.stat().st_size < 10_000:
        raise RuntimeError(f"Video Quality Gate falhou: arquivo ausente ou muito pequeno ({mp4_path})")
    meta = json.loads(subprocess.check_output([
        "ffprobe", "-v", "error", "-select_streams", "v:0",
        "-show_entries", "stream=width,height,codec_name:format=duration",
        "-of", "json", str(mp4_path),
    ], env=WGP_ENV, text=True, timeout=15) or "{}")
    streams = meta.get("streams") or []
    dur = float((meta.get("format") or {}).get("duration") or 0.0)
    if not streams or dur < 0.3:
        raise RuntimeError(f"Video Quality Gate falhou: streams={len(streams)} dur={dur:.2f}s")
    return {
        "duration": round(dur, 2),
        "width": int(streams[0].get("width") or 0),
        "height": int(streams[0].get("height") or 0),
        "bytes": mp4_path.stat().st_size,
    }

t0 = time.time()
p_cuda = subprocess.run([
    WANGP_PY, "-c",
    "import torch; x = torch.randn((2048, 2048), device='cuda', dtype=torch.float16); "
    "y = x @ x; torch.cuda.synchronize(); del x, y; torch.cuda.empty_cache(); "
    "free_b, tot_b = torch.cuda.mem_get_info(0); "
    "print(f'CUDA_L4_OK {torch.cuda.get_device_name(0)} VRAM_livre={free_b/1024**3:.1f}/{tot_b/1024**3:.1f}GB')"
], env=WGP_ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
if p_cuda.returncode != 0:
    raise RuntimeError(p_cuda.stdout)
print(f"[OK] {p_cuda.stdout.strip()} ({time.time() - t0:.2f}s)")

smoke_record = WS / "config/smoke_ok.json"
smoke_record.unlink(missing_ok=True)

if not PRELOAD_WEIGHTS:
    print("[INFO] PRELOAD_WEIGHTS=False: pesos serão baixados sob demanda no primeiro job da VPS.")
else:
    smoke_dir = WS / "tmp/smoke_v4"
    shutil.rmtree(smoke_dir, ignore_errors=True)
    smoke_dir.mkdir(parents=True, exist_ok=True)
    try:
        src_wav = smoke_dir / "src.wav"
        tgt_wav = smoke_dir / "tgt.wav"
        svc_out = smoke_dir / "svc_out"
        svc_out.mkdir(exist_ok=True)
        write_test_wav(src_wav, freq=210.0, dur=1.4)
        write_test_wav(tgt_wav, freq=145.0, dur=1.8)

        t1 = time.time()
        print("Baixando pesos oficiais e validando Seed-VC (--f0-condition False) na L4 GPU...")
        p_svc = subprocess.run([
            SEEDVC_PY, str(WS / "repos/seed-vc/inference.py"),
            "--source", str(src_wav),
            "--target", str(tgt_wav),
            "--output", str(svc_out),
            "--diffusion-steps", "10",
            "--f0-condition", "False",
            "--auto-f0-adjust", "False",
            "--fp16", "True",
        ], cwd=str(WS / "repos/seed-vc"), env=SVC_ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, timeout=600)
        out_wavs = sorted(svc_out.glob("*.wav"))
        if p_svc.returncode != 0 or not out_wavs:
            raise RuntimeError(f"Seed-VC smoke test falhou (rc={p_svc.returncode}):\n{(p_svc.stdout or '')[-1800:]}")
        svc_dur, svc_rms = verify_wav_not_silent(out_wavs[0])
        print(f"[OK] Seed-VC validado ({time.time() - t1:.1f}s) -> {out_wavs[0].name} | dur={svc_dur:.2f}s | rms={svc_rms:.1f}dBFS")

        wgp_meta = {}
        if VIDEO_MODEL != "seed_vc_only":
            t2 = time.time()
            print(f"Baixando pesos INT8 e executando micro-render '{VIDEO_MODEL}' via WanGP na L4...")
            (smoke_dir / "wgp_out").mkdir(parents=True, exist_ok=True)
            smoke_img = smoke_dir / "frame.png"
            subprocess.run([
                "ffmpeg", "-y", "-f", "lavfi", "-i", "testsrc2=size=480x832:duration=1:rate=1",
                "-frames:v", "1", str(smoke_img),
            ], env=WGP_ENV, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)

            smoke_script = (
                "import gc, json, sys, torch\n"
                "from pathlib import Path\n"
                "sys.path.insert(0, '/content/ai_studio_v4/repos/WanGP')\n"
                "from shared.api import init\n"
                "s = init(\n"
                "    root=Path('/content/ai_studio_v4/repos/WanGP'),\n"
                "    config_path=Path('/content/ai_studio_v4/config/wgp_config.json'),\n"
                "    output_dir=Path('/content/ai_studio_v4/tmp/smoke_v4/wgp_out'),\n"
                "    cli_args=['--attention', 'sdpa', '--profile', '4'],\n"
                "    console_output=False,\n"
                ")\n"
                f"model_id = {VIDEO_MODEL!r}\n"
                "cfg = s.get_default_settings(model_id)\n"
                "mdef = s.get_model_def(model_id) or {}\n"
                "schema = (s.get_model_schema(model_id) or {}).get('metadata', {})\n"
                "f_min = int(schema.get('frames_minimum') or mdef.get('frames_minimum') or 17)\n"
                "res = '384x640' if 'ltx2' in model_id else '480x832'\n"
                "cfg['model_type'] = model_id\n"
                "cfg['prompt'] = 'Cinematic still person standing in warm candlelit room, subtle natural breathing motion.'\n"
                "cfg['negative_prompt'] = 'watermark, overlaid text'\n"
                "cfg['resolution'] = res\n"
                "cfg['video_length'] = f_min\n"
                f"cfg['image_start'] = {str(smoke_img)!r}\n"
                "cfg['image_prompt_type'] = 'S'\n"
                "if 'longcat' in model_id:\n"
                f"    cfg['image_refs'] = [{str(smoke_img)!r}]\n"
                "    cfg['video_prompt_type'] = 'KI'\n"
                "    cfg['force_fps'] = ''\n"
                "    cfg['sample_solver'] = 'distill'\n"
                "    cfg['num_inference_steps'] = 8\n"
                "    cfg['guidance_scale'] = 1.0\n"
                "    cfg['audio_guidance_scale'] = 1.0\n"
                f"    cfg['audio_guide'] = {str(tgt_wav)!r}\n"
                "    cfg['audio_prompt_type'] = 'A'\n"
                "elif 'ltx2' in model_id:\n"
                "    cfg['video_prompt_type'] = ''\n"
                "    cfg['image_refs'] = None\n"
                "    cfg['force_fps'] = '24'\n"
                "    cfg['num_inference_steps'] = 8\n"
                f"    cfg['audio_guide'] = {str(tgt_wav)!r}\n"
                "    cfg['audio_prompt_type'] = 'AS'\n"
                "elif model_id in ('infinitetalk', 'multitalk'):\n"
                "    cfg['video_prompt_type'] = ''\n"
                "    cfg['image_refs'] = None\n"
                "    cfg['force_fps'] = ''\n"
                "    cfg['num_inference_steps'] = 6\n"
                f"    cfg['audio_guide'] = {str(tgt_wav)!r}\n"
                "    cfg['audio_prompt_type'] = 'A'\n"
                "else:\n"
                "    cfg['video_prompt_type'] = ''\n"
                "    cfg['image_refs'] = None\n"
                "    cfg['force_fps'] = ''\n"
                "    cfg['audio_prompt_type'] = ''\n"
                "    cfg['num_inference_steps'] = 6\n"
                "cfg['overrides'] = (\n"
                "    \"attention_mode='sdpa', memory_profile=4, \"\n"
                "    \"transformer_quantization='int8', text_encoder_quantization='int8'\"\n"
                ")\n"
                "job = s.submit_task(cfg)\n"
                "r = job.result()\n"
                "assert r.success and r.generated_files, 'WanGP falhou: ' + '; '.join(str(e) for e in (r.errors or []))\n"
                "p = Path(r.generated_files[0])\n"
                "assert p.is_file() and p.stat().st_size > 10000, f'Arquivo MP4 inválido: {p}'\n"
                "gc.collect()\n"
                "torch.cuda.synchronize()\n"
                "torch.cuda.empty_cache()\n"
                "free_b, tot_b = torch.cuda.mem_get_info(0)\n"
                "print('WANGP_SMOKE_JSON ' + json.dumps({'path': str(p), 'model': model_id, 'res': res, 'frames': f_min, 'vram_free_gb': round(free_b/1024**3, 1), 'vram_total_gb': round(tot_b/1024**3, 1)}))\n"
            )
            smoke_py = smoke_dir / "run_smoke_wgp.py"
            smoke_py.write_text(smoke_script, encoding="utf-8")
            p_wgp = subprocess.run([WANGP_PY, str(smoke_py)], env=WGP_ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, timeout=1500)
            json_line = next((ln for ln in (p_wgp.stdout or "").splitlines() if ln.startswith("WANGP_SMOKE_JSON ")), "")
            if p_wgp.returncode != 0 or not json_line:
                tail = "\n".join((p_wgp.stdout or "").splitlines()[-65:])
                raise RuntimeError(f"WanGP smoke test '{VIDEO_MODEL}' falhou:\n{tail}")
            wgp_res = json.loads(json_line.split(" ", 1)[1])
            wgp_meta = verify_mp4_valid(Path(wgp_res["path"]))
            wgp_meta.update({"res": wgp_res["res"], "frames": wgp_res["frames"], "vram_free_gb": wgp_res["vram_free_gb"]})
            print(
                f"[OK] WanGP '{VIDEO_MODEL}' validado ({time.time() - t2:.1f}s) -> "
                f"{wgp_meta['width']}x{wgp_meta['height']} dur={wgp_meta['duration']}s "
                f"bytes={wgp_meta['bytes']} VRAM_livre={wgp_res['vram_free_gb']}/{wgp_res['vram_total_gb']}GB"
            )

        smoke_record.write_text(json.dumps({
            "ok": True,
            "video_model": VIDEO_MODEL,
            "seed_vc": {"duration": svc_dur, "rms_dbfs": svc_rms},
            "wangp": wgp_meta,
            "timestamp": int(time.time()),
        }, indent=2), encoding="utf-8")
    finally:
        shutil.rmtree(smoke_dir, ignore_errors=True)

    disk_after = round(shutil.disk_usage("/content").free / (1024 ** 3), 1)
    print(f"Pré-download + Smoke Test L4 concluídos com sucesso! Disco /content livre: {disk_after} GB")


In [ ]:
# @title 4. Iniciar Servidor Worker Cinema Engine v4.0 + Túnel Cloudflare (Watchdog Auto-Recover)
import json, re, subprocess, sys, time, urllib.request, warnings
from pathlib import Path

warnings.filterwarnings("ignore")

WS = Path("/content/ai_studio_v4")
if str(WS) not in sys.path:
    sys.path.insert(0, str(WS))
from clean_env import build_clean_env  # noqa: E402

env_cfg = json.loads((WS / "config/worker_env.json").read_text(encoding="utf-8"))

SERVER_CODE = '#!/usr/bin/env python3\n"""Google Colab Pro L4 (24GB VRAM) Cinema Engine v4.0 Worker Server.\n\nZero-Drive, Clean-Slate Architecture (v4.0.3-clean-l4):\n1. Hermetic environment sanitization at process bootstrap and every subprocess call\n   (eliminates Jupyter MPLBACKEND/PYTHONPATH/JPY_* leaks while preserving CUDA/NVIDIA\n   driver libs and COLAB_AUTH_TOKEN).\n2. Dual isolated uv Python 3.10 virtual environments under /content/ai_studio_v4/envs/:\n   - envs/seedvc : numpy==1.26.4 + transformers==4.46.3 + torchcodec + TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1\n   - envs/wangp  : FastAPI + WanGP v20.00 shared/api.py persistent session (INT8/SDPA/Profile-4)\n3. Model-aware WanGP settings verified against upstream handlers:\n   - ltx2_25_22B_distilled : vae_block_size=64, image_start (S), audio_guide (AS), force_fps=\'24\'\n   - longcat_avatar_v1_6 / v1_5 : image_start (S) + anchor image_refs (KI), audio_guide (A), distill 8-step\n   - infinitetalk : image_start (S), audio_guide (A), 480x832\n   - i2v_2_2 : image_start (S), 480x832\n4. Real Audio Quality Gate (Seed-VC) and Video Quality Gate (WanGP) — zero silent/static fallbacks.\n5. Thread-safe async Job Queue (/jobs/animate_scene + /jobs/{job_id}) immune to Cloudflare 524.\n6. Structured stdout + file logging (/tmp/colab_v4_server.log), zero stderr spam.\n"""\n\nfrom __future__ import annotations\n\nimport array\nimport base64\nimport gc\nimport hmac\nimport inspect\nimport json\nimport math\nimport os\nimport shutil\nimport subprocess\nimport sys\nimport threading\nimport time\nimport uuid\nimport wave\nfrom pathlib import Path\nfrom typing import Any, Dict, Optional\n\n# ---------------------------------------------------------------------------\n# 1. Hermetic Environment Sanitizer (MUST run BEFORE importing torch / WanGP)\n# ---------------------------------------------------------------------------\n\nTOXIC_EXACT_VARS = {\n    "MPLBACKEND",\n    "PYTHONHOME",\n    "PYTHONPATH",\n    "PYTHONSTARTUP",\n    "PYTHONUSERBASE",\n    "PYTHONEXECUTABLE",\n    "CONDA_PREFIX",\n    "CONDA_DEFAULT_ENV",\n    "CONDA_SHLVL",\n    "PIP_CONSTRAINT",\n    "PIP_REQUIRE_VIRTUALENV",\n    "UV_PYTHON",\n    "JPY_PARENT_PID",\n    "JPY_SESSION_NAME",\n    "JPY_API_TOKEN",\n    "IPYTHONDIR",\n    "DISPLAY",\n    "LD_PRELOAD",\n}\n\nTOXIC_PREFIXES = (\n    "JPY_",\n    "JUPYTER_",\n    "IPYTHON_",\n    "COLAB_",\n    "TBE_",\n)\n\n# Never strip application secrets that share the COLAB_ prefix!\nPRESERVED_EXACT_VARS = {\n    "COLAB_AUTH_TOKEN",\n}\n\nWORKSPACE_ROOT = Path(os.environ.get("AI_STUDIO_ROOT", "/content/ai_studio_v4"))\nCACHE_DIR = WORKSPACE_ROOT / "cache"\nHF_CACHE_DIR = Path(os.environ.get("HF_HOME", str(WORKSPACE_ROOT / "hf_cache")))\n\n\ndef _augment_cuda_ld_library_path(existing: str) -> str:\n    """Append existing host CUDA/NVIDIA library paths without clobbering existing entries."""\n    candidates: list[str] = []\n    for base in ("/usr/lib64-nvidia", "/usr/local/cuda/lib64"):\n        if Path(base).is_dir():\n            candidates.append(base)\n    usr_local = Path("/usr/local")\n    if usr_local.is_dir():\n        for p in sorted(usr_local.glob("cuda-12*/targets/x86_64-linux/lib")):\n            if p.is_dir():\n                candidates.append(str(p))\n    parts = [x for x in (existing or "").split(":") if x]\n    for c in candidates:\n        if c not in parts:\n            parts.append(c)\n    return ":".join(parts)\n\n\ndef build_clean_env(\n    venv_dir: Optional[Path] = None,\n    extra: Optional[Dict[str, str]] = None,\n    base_env: Optional[Dict[str, str]] = None,\n) -> Dict[str, str]:\n    """Build a deterministic, Jupyter-free environment preserving CUDA/NVIDIA & auth secrets."""\n    src = dict(os.environ if base_env is None else base_env)\n    cleaned: Dict[str, str] = {}\n    for k, v in src.items():\n        if k in PRESERVED_EXACT_VARS:\n            cleaned[k] = v\n            continue\n        if k in TOXIC_EXACT_VARS or k == "VIRTUAL_ENV":\n            continue\n        if any(k.startswith(pref) for pref in TOXIC_PREFIXES):\n            continue\n        cleaned[k] = v\n\n    ld_path = _augment_cuda_ld_library_path(cleaned.get("LD_LIBRARY_PATH", ""))\n    if ld_path:\n        cleaned["LD_LIBRARY_PATH"] = ld_path\n\n    forced = {\n        "MPLBACKEND": "Agg",\n        "QT_QPA_PLATFORM": "offscreen",\n        "PYGAME_HIDE_SUPPORT_PROMPT": "1",\n        "TOKENIZERS_PARALLELISM": "false",\n        "PIP_NO_INPUT": "1",\n        "PIP_NO_WARN_SCRIPT_LOCATION": "1",\n        "UV_NO_PROGRESS": "1",\n        "UV_LINK_MODE": "copy",\n        "PYTHONWARNINGS": "ignore",\n        "PYTHONUNBUFFERED": "1",\n        "PYTHONNOUSERSITE": "1",\n        "DEBIAN_FRONTEND": "noninteractive",\n        "HF_HUB_DISABLE_PROGRESS_BARS": "1",\n        "HF_HUB_VERBOSITY": "error",\n        "HF_HOME": str(HF_CACHE_DIR),\n        "HF_HUB_CACHE": str(HF_CACHE_DIR / "hub"),\n        "TRITON_CACHE_DIR": str(CACHE_DIR / "triton"),\n        "TORCHINDUCTOR_CACHE_DIR": str(CACHE_DIR / "inductor"),\n        "MPLCONFIGDIR": str(CACHE_DIR / "matplotlib"),\n        "XDG_CACHE_HOME": str(CACHE_DIR / "xdg"),\n        "WAN2GP_DISABLE_AUDIO": "1",\n        "TF_CPP_MIN_LOG_LEVEL": "3",\n        "LC_ALL": "C.UTF-8",\n        "LANG": "C.UTF-8",\n    }\n    cleaned.update(forced)\n\n    if venv_dir is not None:\n        vbin = str(Path(venv_dir) / "bin")\n        cleaned["VIRTUAL_ENV"] = str(venv_dir)\n        old_path = cleaned.get("PATH", "/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin")\n        parts = [p for p in old_path.split(":") if p and p != vbin]\n        cleaned["PATH"] = ":".join([vbin] + parts)\n\n    if extra:\n        for ek, ev in extra.items():\n            if ev is not None:\n                cleaned[ek] = str(ev)\n\n    return cleaned\n\n\ndef _sanitize_self_env() -> None:\n    """Neutralize inherited Colab/Jupyter variables in the server process without wiping auth."""\n    for k in list(os.environ.keys()):\n        if k in PRESERVED_EXACT_VARS:\n            continue\n        if k in TOXIC_EXACT_VARS or any(k.startswith(pref) for pref in TOXIC_PREFIXES):\n            os.environ.pop(k, None)\n    clean_defaults = build_clean_env(base_env=dict(os.environ))\n    for k, v in clean_defaults.items():\n        if k != "PATH":\n            os.environ[k] = v\n\n\n_sanitize_self_env()\n\nimport torch  # noqa: E402\nfrom fastapi import FastAPI, Header, HTTPException  # noqa: E402\nfrom pydantic import BaseModel  # noqa: E402\nimport uvicorn  # noqa: E402\n\n\nSERVER_VERSION = "4.0.3-clean-l4"  # Must start with "4.0" (vps_novela_runner.py L2380)\nREPOS_DIR = WORKSPACE_ROOT / "repos"\nCONFIG_DIR = WORKSPACE_ROOT / "config"\nOUTPUTS_DIR = WORKSPACE_ROOT / "outputs" / "wangp"\n\nCOLAB_AUTH_TOKEN = os.environ.get("COLAB_AUTH_TOKEN", "").strip()\nDEFAULT_VIDEO_MODEL = os.environ.get("AI_STUDIO_VIDEO_MODEL", "ltx2_25_22B_distilled").strip()\n\nSEED_VC_VENV = WORKSPACE_ROOT / "envs" / "seedvc"\nSEED_VC_PYTHON = Path(os.environ.get("AI_STUDIO_SEEDVC_PY", str(SEED_VC_VENV / "bin" / "python")))\nSEED_VC_DIR = Path(os.environ.get("AI_STUDIO_SEEDVC_DIR", str(REPOS_DIR / "seed-vc")))\nWANGP_DIR = Path(os.environ.get("AI_STUDIO_WANGP_DIR", str(REPOS_DIR / "WanGP")))\n\n# Verified against WanGP/shared/utils/generation_overrides.py OVERRIDE_DEFAULTS\nWANGP_L4_OVERRIDES = (\n    "attention_mode=\'sdpa\', memory_profile=4, "\n    "transformer_quantization=\'int8\', text_encoder_quantization=\'int8\'"\n)\n\nGPU_LOCK = threading.Lock()\nJOBS_LOCK = threading.Lock()\nJOBS: Dict[str, Dict[str, Any]] = {}\n_WANGP_SESSION = None\n_LAST_LOADED_MODEL: Optional[str] = None\n\n\nBACKEND_MODEL_MAP: Dict[str, str] = {\n    "auto": DEFAULT_VIDEO_MODEL,\n    "wangp_wan22_s2v": DEFAULT_VIDEO_MODEL,\n    "wan22_s2v_v4": DEFAULT_VIDEO_MODEL,\n    "ltx25_v4": "ltx2_25_22B_distilled",\n    "ltx25_distilled": "ltx2_25_22B_distilled",\n    "ltx2_25_22b_distilled": "ltx2_25_22B_distilled",\n    "longcat_avatar": "longcat_avatar_v1_6",\n    "longcat_avatar_v1_6": "longcat_avatar_v1_6",\n    "longcat_avatar_v1_5": "longcat_avatar_v1_5",\n    "infinitetalk": "infinitetalk",\n    "multitalk": "multitalk",\n    "i2v_2_2": "i2v_2_2",\n}\n\n\ndef _cleanup_vram() -> None:\n    gc.collect()\n    if torch.cuda.is_available():\n        try:\n            torch.cuda.synchronize()\n        except Exception:\n            pass\n        try:\n            torch.cuda.empty_cache()\n            torch.cuda.ipc_collect()\n        except Exception:\n            pass\n\n\ndef _gpu_info() -> Dict[str, Any]:\n    if not torch.cuda.is_available():\n        return {"gpu": "CPU", "vram_gb": 0.0, "vram_free_gb": 0.0, "cuda_ok": False}\n    free_b, total_b = torch.cuda.mem_get_info(0)\n    return {\n        "gpu": torch.cuda.get_device_name(0),\n        "vram_gb": round(total_b / (1024 ** 3), 1),\n        "vram_free_gb": round(free_b / (1024 ** 3), 1),\n        "cuda_ok": True,\n        "torch_version": torch.__version__,\n        "cuda_version": getattr(torch.version, "cuda", ""),\n    }\n\n\ndef _smi_used_gb() -> float:\n    try:\n        out = subprocess.check_output(\n            ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],\n            text=True,\n            env=build_clean_env(),\n            timeout=5,\n        ).strip().splitlines()\n        if out:\n            return round(float(out[0].strip()) / 1024.0, 2)\n    except Exception:\n        pass\n    if torch.cuda.is_available():\n        return round(torch.cuda.max_memory_allocated() / (1024 ** 3), 2)\n    return 0.0\n\n\ndef _require_token(token: str, require_configured: bool = False) -> None:\n    if require_configured and not COLAB_AUTH_TOKEN:\n        raise HTTPException(status_code=403, detail="Rota administrativa exige COLAB_AUTH_TOKEN configurado.")\n    if COLAB_AUTH_TOKEN and not hmac.compare_digest(token or "", COLAB_AUTH_TOKEN):\n        raise HTTPException(status_code=401, detail="X-Colab-Token inválido")\n\n\ndef _probe_dur(path: Path) -> float:\n    out = subprocess.check_output(\n        [\n            "ffprobe",\n            "-v",\n            "error",\n            "-show_entries",\n            "format=duration",\n            "-of",\n            "default=noprint_wrappers=1:nokey=1",\n            str(path),\n        ],\n        text=True,\n        env=build_clean_env(),\n        timeout=15,\n    ).strip()\n    return max(0.0, float(out))\n\n\ndef _validate_wav_quality(\n    wav_path: Path,\n    min_dur: float = 0.25,\n    min_rms_dbfs: float = -55.0,\n) -> Dict[str, float]:\n    """Real Audio Quality Gate: rejects empty, truncated, or silent WAV outputs."""\n    if not wav_path.is_file() or wav_path.stat().st_size < 1024:\n        raise RuntimeError(f"Audio Quality Gate falhou: arquivo ausente ou vazio ({wav_path.name})")\n    with wave.open(str(wav_path), "rb") as wf:\n        n_frames = wf.getnframes()\n        sr = wf.getframerate()\n        sampwidth = wf.getsampwidth()\n        if sr <= 0 or n_frames <= 0:\n            raise RuntimeError("Audio Quality Gate falhou: WAV sem frames válidos.")\n        dur = n_frames / float(sr)\n        if dur < min_dur:\n            raise RuntimeError(f"Audio Quality Gate falhou: duração {dur:.3f}s abaixo do mínimo ({min_dur}s).")\n        if sampwidth != 2:\n            raise RuntimeError(f"Audio Quality Gate falhou: esperado PCM 16-bit, recebido {sampwidth * 8}-bit.")\n        raw = wf.readframes(n_frames)\n\n    samples = array.array("h")\n    samples.frombytes(raw)\n    total = len(samples)\n    if total == 0:\n        raise RuntimeError("Audio Quality Gate falhou: 0 amostras PCM.")\n\n    step = max(1, total // 120_000)\n    sampled = samples[::step]\n    peak = max(abs(x) for x in sampled)\n    rms = math.sqrt(sum((x / 32768.0) ** 2 for x in sampled) / len(sampled))\n    rms_dbfs = 20.0 * math.log10(max(rms, 1e-12))\n    peak_dbfs = 20.0 * math.log10(max(peak / 32768.0, 1e-12))\n\n    if rms_dbfs < min_rms_dbfs or peak < 64:\n        raise RuntimeError(\n            f"Audio Quality Gate falhou (silêncio/sinal nulo): dur={dur:.2f}s "\n            f"rms={rms_dbfs:.1f}dBFS peak={peak_dbfs:.1f}dBFS"\n        )\n    return {\n        "duration": round(dur, 2),\n        "rms_dbfs": round(rms_dbfs, 1),\n        "peak_dbfs": round(peak_dbfs, 1),\n    }\n\n\ndef _validate_video_output(mp4_path: Path, min_dur: float = 0.3, min_bytes: int = 10_000) -> Dict[str, Any]:\n    """Real Video Quality Gate: guarantees valid playable H.264 stream with real duration."""\n    if not mp4_path.is_file() or mp4_path.stat().st_size < min_bytes:\n        sz = mp4_path.stat().st_size if mp4_path.is_file() else 0\n        raise RuntimeError(f"Video Quality Gate falhou: MP4 ausente ou pequeno demais ({sz} bytes).")\n    out = subprocess.check_output(\n        [\n            "ffprobe",\n            "-v",\n            "error",\n            "-select_streams",\n            "v:0",\n            "-show_entries",\n            "stream=width,height,codec_name:format=duration",\n            "-of",\n            "json",\n            str(mp4_path),\n        ],\n        text=True,\n        env=build_clean_env(),\n        timeout=15,\n    )\n    meta = json.loads(out or "{}")\n    streams = meta.get("streams") or []\n    if not streams:\n        raise RuntimeError("Video Quality Gate falhou: nenhum stream de vídeo válido no MP4.")\n    dur = float((meta.get("format") or {}).get("duration") or 0.0)\n    if dur < min_dur:\n        raise RuntimeError(f"Video Quality Gate falhou: duração {dur:.3f}s abaixo do mínimo ({min_dur}s).")\n    return {\n        "duration": round(dur, 2),\n        "width": int(streams[0].get("width") or 0),\n        "height": int(streams[0].get("height") or 0),\n        "bytes": mp4_path.stat().st_size,\n    }\n\n\ndef _mux_no_freeze(\n    raw_mp4: Path,\n    wav_path: Optional[Path],\n    out_mp4: Path,\n    width: int,\n    height: int,\n    fps: int = 24,\n) -> None:\n    vf = (\n        f"scale={width}:{height}:force_original_aspect_ratio=increase:flags=lanczos,"\n        f"crop={width}:{height},setsar=1,fps={fps},format=yuv420p"\n    )\n    vid_dur = max(0.3, _probe_dur(raw_mp4))\n    if wav_path and wav_path.is_file():\n        aud_dur = max(0.3, _probe_dur(wav_path))\n        target_dur = min(vid_dur, aud_dur)\n        cmd = [\n            "ffmpeg", "-y", "-i", str(raw_mp4), "-i", str(wav_path),\n            "-map", "0:v:0", "-map", "1:a:0",\n            "-vf", vf, "-t", f"{target_dur:.3f}",\n            "-c:v", "libx264", "-preset", "fast", "-crf", "17",\n            "-c:a", "aac", "-b:a", "192k", "-ar", "48000", "-ac", "2",\n            "-movflags", "+faststart",\n            str(out_mp4),\n        ]\n    else:\n        cmd = [\n            "ffmpeg", "-y", "-i", str(raw_mp4),\n            "-vf", vf,\n            "-c:v", "libx264", "-preset", "fast", "-crf", "17",\n            "-c:a", "aac", "-b:a", "192k", "-ar", "48000", "-ac", "2",\n            "-movflags", "+faststart",\n            str(out_mp4),\n        ]\n    proc = subprocess.run(cmd, env=build_clean_env(), capture_output=True, text=True, timeout=120)\n    if proc.returncode != 0:\n        raise RuntimeError(f"FFmpeg mux falhou (rc={proc.returncode}): {(proc.stderr or proc.stdout)[-400:]}")\n\n\ndef _get_wangp_session():\n    global _WANGP_SESSION\n    if _WANGP_SESSION is not None:\n        return _WANGP_SESSION\n    if not (WANGP_DIR / "shared" / "api.py").is_file():\n        raise RuntimeError(f"WanGP não instalado em {WANGP_DIR}")\n    CONFIG_DIR.mkdir(parents=True, exist_ok=True)\n    OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)\n    if str(WANGP_DIR) not in sys.path:\n        sys.path.insert(0, str(WANGP_DIR))\n\n    from shared.api import init as wangp_init  # type: ignore\n\n    init_kwargs: Dict[str, Any] = {\n        "root": WANGP_DIR,\n        "config_path": CONFIG_DIR / "wgp_config.json",\n        "output_dir": OUTPUTS_DIR,\n        "cli_args": ["--attention", "sdpa", "--profile", "4"],\n        "console_output": False,\n    }\n    try:\n        sig = inspect.signature(wangp_init)\n        accepted = set(sig.parameters.keys())\n        if not any(p.kind == inspect.Parameter.VAR_KEYWORD for p in sig.parameters.values()):\n            init_kwargs = {k: v for k, v in init_kwargs.items() if k in accepted}\n    except Exception:\n        pass\n\n    _WANGP_SESSION = wangp_init(**init_kwargs)\n    return _WANGP_SESSION\n\n\ndef _resolve_model_type(requested_backend: str, has_audio: bool) -> str:\n    key = (requested_backend or "auto").strip().lower()\n    if key == "seed_vc_only":\n        raise RuntimeError("Este worker foi iniciado em modo seed_vc_only (sem modelo de vídeo).")\n    resolved = BACKEND_MODEL_MAP.get(key) or DEFAULT_VIDEO_MODEL\n    if resolved == "seed_vc_only":\n        raise RuntimeError("Worker configurado apenas para Seed-VC (VIDEO_MODEL=seed_vc_only).")\n    if not has_audio and resolved in ("infinitetalk", "multitalk", "longcat_avatar_v1_5", "longcat_avatar_v1_6"):\n        return "i2v_2_2"\n    return resolved\n\n\ndef _resolve_model_resolution(model_type: str, width: int, height: int) -> str:\n    portrait = height >= width\n    if model_type in ("infinitetalk", "multitalk") or "longcat" in model_type:\n        return "480x832" if portrait else "832x480"\n    if "ltx2" in model_type:\n        # Verified in models/ltx2/ltx2_handler.py: vae_block_size = 64\n        w = max(256, min(704 if portrait else 1280, (int(width) // 64) * 64))\n        h = max(256, min(1280 if portrait else 704, (int(height) // 64) * 64))\n        return f"{w}x{h}"\n    return "480x832" if portrait else "832x480"\n\n\ndef _build_wangp_task_settings(\n    base_settings: Dict[str, Any],\n    model_type: str,\n    model_def: Dict[str, Any],\n    schema_meta: Dict[str, Any],\n    *,\n    prompt: str,\n    negative_prompt: str,\n    img_path: Path,\n    wav_path: Optional[Path],\n    dur_target: float,\n    width: int,\n    height: int,\n    fps: int,\n    steps: int,\n    guidance_scale: float,\n    seed: int,\n    resolution_override: Optional[str] = None,\n    force_min_frames: bool = False,\n) -> Dict[str, Any]:\n    """Assemble validated per-family WanGP settings matching upstream handlers."""\n    settings = dict(base_settings or {})\n    native_fps = int(model_def.get("fps") or fps or 24)\n    f_min = int(schema_meta.get("frames_minimum") or model_def.get("frames_minimum") or 17)\n    f_step = int(schema_meta.get("frames_steps") or model_def.get("frames_steps") or 8)\n    f_off = int(model_def.get("frames_offset") or 1)\n\n    if force_min_frames:\n        aligned_frames = f_min\n    else:\n        eff_fps = native_fps if ("longcat" in model_type or model_type in ("infinitetalk", "multitalk")) else fps\n        raw_frames = max(f_min, int(math.ceil(dur_target * eff_fps)) + 4)\n        steps_n = max(0, (raw_frames - f_off + f_step - 1) // f_step)\n        aligned_frames = max(f_min, steps_n * f_step + f_off)\n\n    settings["model_type"] = model_type\n    settings["prompt"] = prompt\n    settings["negative_prompt"] = negative_prompt\n    settings["resolution"] = resolution_override or _resolve_model_resolution(model_type, width, height)\n    settings["video_length"] = aligned_frames\n    settings["seed"] = int(seed)\n\n    # All 4 families (LTX-2.5, LongCat Avatar 1.5/1.6, InfiniteTalk, Wan2.2 I2V) accept "S" + image_start\n    settings["image_start"] = str(img_path)\n    settings["image_prompt_type"] = "S"\n\n    if "longcat" in model_type:\n        # LongCat Avatar supports anchor reference image (KI) preventing identity drift across windows\n        settings["image_refs"] = [str(img_path)]\n        settings["video_prompt_type"] = "KI"\n        settings["force_fps"] = ""  # preserve native 25fps audio_stride=1; FFmpeg muxes cleanly to target fps\n        settings["sample_solver"] = "distill"\n        settings["num_inference_steps"] = 8\n        settings["guidance_scale"] = 1.0\n        settings["audio_guidance_scale"] = 1.0\n        if wav_path is not None:\n            settings["audio_guide"] = str(wav_path)\n            settings["audio_prompt_type"] = "A"\n        else:\n            settings["audio_guide"] = None\n            settings["audio_prompt_type"] = ""\n    elif "ltx2" in model_type:\n        settings["video_prompt_type"] = ""\n        settings["image_refs"] = None\n        settings["force_fps"] = str(int(fps or 24))\n        if "distilled" in model_type:\n            settings["num_inference_steps"] = 8\n        else:\n            settings["num_inference_steps"] = max(4, int(steps))\n            settings["guidance_scale"] = float(guidance_scale)\n        if wav_path is not None:\n            settings["audio_guide"] = str(wav_path)\n            settings["audio_prompt_type"] = "AS"\n        else:\n            settings["audio_guide"] = None\n            settings["audio_prompt_type"] = ""\n    elif model_type in ("infinitetalk", "multitalk"):\n        settings["video_prompt_type"] = ""\n        settings["image_refs"] = None\n        settings["force_fps"] = ""\n        settings["num_inference_steps"] = max(4, int(steps))\n        settings["guidance_scale"] = float(guidance_scale)\n        if wav_path is not None:\n            settings["audio_guide"] = str(wav_path)\n            settings["audio_prompt_type"] = "A"\n        else:\n            settings["audio_guide"] = None\n            settings["audio_prompt_type"] = ""\n    else:\n        settings["video_prompt_type"] = ""\n        settings["image_refs"] = None\n        settings["force_fps"] = ""\n        settings["audio_guide"] = None\n        settings["audio_prompt_type"] = ""\n        settings["num_inference_steps"] = max(4, int(steps))\n        settings["guidance_scale"] = float(guidance_scale)\n\n    settings["overrides"] = WANGP_L4_OVERRIDES\n    return settings\n\n\ndef _evict_finished_jobs_locked(max_finished: int = 16) -> None:\n    """Never evict queued or processing jobs; prune oldest completed/error jobs."""\n    finished = [\n        (j.get("created_at", 0.0), jid)\n        for jid, j in JOBS.items()\n        if j.get("status") in ("completed", "error")\n    ]\n    if len(finished) <= max_finished:\n        return\n    finished.sort(key=lambda item: item[0])\n    for _, jid in finished[: len(finished) - max_finished]:\n        JOBS.pop(jid, None)\n\n\ndef _read_smoke_status() -> bool:\n    smoke_meta = CONFIG_DIR / "smoke_ok.json"\n    if not smoke_meta.is_file():\n        return False\n    try:\n        data = json.loads(smoke_meta.read_text(encoding="utf-8"))\n        return bool(data.get("ok")) and data.get("video_model") == DEFAULT_VIDEO_MODEL\n    except Exception:\n        return False\n\n\napp = FastAPI(title="AI Media Studio Colab Cinema Engine v4.0", version=SERVER_VERSION)\n\n\nclass ExecReq(BaseModel):\n    cmd: str\n    timeout: int = 600\n\n\nclass VoiceConvertReq(BaseModel):\n    guide_audio_b64: str\n    target_timbre_b64: str\n    character_id: str = "dona_socorro"\n    diffusion_steps: int = 25\n    semi_tone_shift: int = 0\n    f0_condition: bool = False\n\n\nclass AnimateSceneReq(BaseModel):\n    image_b64: str\n    audio_b64: str = ""\n    prompt: str\n    negative_prompt: str = "static portrait, frozen posture, distorted hands, extra fingers, watermark, overlaid text"\n    backend: str = "auto"\n    steps: int = 8\n    guidance_scale: float = 3.5\n    duration_sec: float = 3.2\n    width: int = 720\n    height: int = 1280\n    fps: int = 24\n    seed: int = 42\n\n\n@app.get("/health")\ndef health(x_colab_token: str = Header(default="")):\n    _require_token(x_colab_token, require_configured=False)\n    info = _gpu_info()\n    disk_free_gb = 0.0\n    try:\n        st = shutil.disk_usage(str(WORKSPACE_ROOT if WORKSPACE_ROOT.exists() else Path("/content")))\n        disk_free_gb = round(st.free / (1024 ** 3), 1)\n    except Exception:\n        pass\n    seed_vc_ready = SEED_VC_PYTHON.is_file() and (SEED_VC_DIR / "inference.py").is_file()\n    wangp_installed = (WANGP_DIR / "wgp.py").is_file() and (WANGP_DIR / "shared" / "api.py").is_file()\n    wangp_ready = wangp_installed and DEFAULT_VIDEO_MODEL != "seed_vc_only"\n    ltx2_ready = wangp_ready and "ltx2" in DEFAULT_VIDEO_MODEL\n    longcat_ready = wangp_ready and "longcat" in DEFAULT_VIDEO_MODEL\n    is_ready = bool(info["cuda_ok"] and seed_vc_ready and (wangp_ready or DEFAULT_VIDEO_MODEL == "seed_vc_only"))\n    engines = [\n        name\n        for name, enabled in (\n            ("wan22_s2v", wangp_ready),\n            ("ltx25", ltx2_ready),\n            ("longcat", longcat_ready),\n            ("seed_vc", seed_vc_ready),\n        )\n        if enabled\n    ]\n    with JOBS_LOCK:\n        active_jobs = sum(1 for j in JOBS.values() if j.get("status") in ("queued", "processing"))\n    return {\n        "ok": True,\n        "ready": is_ready,\n        "version": SERVER_VERSION,\n        **info,\n        "default_video_model": DEFAULT_VIDEO_MODEL,\n        "loaded_video_model": _LAST_LOADED_MODEL,\n        "workspace_root": str(WORKSPACE_ROOT),\n        "cache_free_gb": disk_free_gb,\n        "drive_mounted": False,\n        "env_sanitized": (\n            os.environ.get("MPLBACKEND") == "Agg"\n            and "PYTHONPATH" not in os.environ\n            and "JPY_PARENT_PID" not in os.environ\n        ),\n        "smoke_verified": _read_smoke_status(),\n        "seed_vc_ready": seed_vc_ready,\n        "wangp_ready": wangp_ready,\n        "ltx2_ready": ltx2_ready,\n        "longcat_ready": longcat_ready,\n        "engines": engines,\n        "wangp_session_warm": _WANGP_SESSION is not None,\n        "active_jobs": active_jobs,\n    }\n\n\n@app.post("/admin/exec")\ndef admin_exec(req: ExecReq, x_colab_token: str = Header(default="")):\n    _require_token(x_colab_token, require_configured=True)\n    proc = subprocess.run(\n        req.cmd,\n        shell=True,\n        env=build_clean_env(),\n        capture_output=True,\n        text=True,\n        timeout=req.timeout,\n    )\n    return {"rc": proc.returncode, "stdout": proc.stdout[-60000:], "stderr": proc.stderr[-25000:]}\n\n\n@app.post("/voice-convert")\ndef voice_convert(req: VoiceConvertReq, x_colab_token: str = Header(default="")):\n    _require_token(x_colab_token, require_configured=False)\n    inf_py = SEED_VC_DIR / "inference.py"\n    if not inf_py.is_file() or not SEED_VC_PYTHON.is_file():\n        raise HTTPException(status_code=503, detail=f"Seed-VC não encontrado em {SEED_VC_DIR}")\n\n    job_dir = WORKSPACE_ROOT / "tmp" / f"svc_v4_{uuid.uuid4().hex[:8]}"\n    job_dir.mkdir(parents=True, exist_ok=True)\n    t0 = time.time()\n    with GPU_LOCK:\n        try:\n            g_wav = job_dir / "source.wav"\n            t_wav = job_dir / "target.wav"\n            out_dir = job_dir / "out"\n            out_dir.mkdir(exist_ok=True)\n            out_wav = job_dir / "converted_48k.wav"\n            g_wav.write_bytes(base64.b64decode(req.guide_audio_b64))\n            t_wav.write_bytes(base64.b64decode(req.target_timbre_b64))\n\n            # Verified in Plachtaa/seed-vc/inference.py L325-343: auto_f0_adjust applies inside if f0_condition\n            auto_f0 = "True" if (req.f0_condition and int(req.semi_tone_shift) == 0) else "False"\n            cmd = [\n                str(SEED_VC_PYTHON),\n                str(inf_py),\n                "--source", str(g_wav),\n                "--target", str(t_wav),\n                "--output", str(out_dir),\n                "--diffusion-steps", str(max(1, int(req.diffusion_steps))),\n                "--semi-tone-shift", str(int(req.semi_tone_shift)),\n                "--f0-condition", "True" if req.f0_condition else "False",\n                "--auto-f0-adjust", auto_f0,\n                "--fp16", "True",\n            ]\n            # Plachtaa/seed-vc (PyTorch 2.4 era) calls bare torch.load(); allow legacy checkpoints on PyTorch 2.6+/2.10\n            svc_env = build_clean_env(\n                venv_dir=SEED_VC_VENV,\n                extra={\n                    "HF_HOME": str(HF_CACHE_DIR),\n                    "TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD": "1",\n                },\n            )\n            proc = subprocess.run(\n                cmd,\n                cwd=str(SEED_VC_DIR),\n                env=svc_env,\n                capture_output=True,\n                text=True,\n                timeout=240,\n            )\n            gen = next(iter(sorted(out_dir.glob("*.wav"))), None)\n            if proc.returncode != 0 or not gen or not gen.is_file():\n                diag = (proc.stderr or proc.stdout or "").strip()[-600:]\n                raise RuntimeError(f"Seed-VC falhou (rc={proc.returncode}): {diag}")\n\n            ff = subprocess.run(\n                ["ffmpeg", "-y", "-i", str(gen), "-ar", "48000", "-ac", "2", "-c:a", "pcm_s16le", str(out_wav)],\n                env=build_clean_env(),\n                capture_output=True,\n                text=True,\n                timeout=60,\n            )\n            if ff.returncode != 0:\n                raise RuntimeError(f"FFmpeg resample falhou: {(ff.stderr or ff.stdout)[-400:]}")\n\n            # Real Audio Quality Gate (zero silent/corrupted fallback)\n            aqg = _validate_wav_quality(out_wav, min_dur=0.25, min_rms_dbfs=-55.0)\n            return {\n                "ok": True,\n                "engine": "seed-vc-v4-real",\n                "elapsed_s": round(time.time() - t0, 2),\n                "duration": aqg["duration"],\n                "rms_dbfs": aqg["rms_dbfs"],\n                "peak_dbfs": aqg["peak_dbfs"],\n                "vram_used_gb": _smi_used_gb(),\n                "character_id": req.character_id,\n                "audio_b64": base64.b64encode(out_wav.read_bytes()).decode("ascii"),\n            }\n        except HTTPException:\n            raise\n        except Exception as exc:\n            raise HTTPException(status_code=500, detail=str(exc)[:600])\n        finally:\n            shutil.rmtree(job_dir, ignore_errors=True)\n            _cleanup_vram()\n\n\ndef _execute_scene_job(job_id: str, req: AnimateSceneReq) -> None:\n    global _LAST_LOADED_MODEL\n    job_dir = WORKSPACE_ROOT / "tmp" / f"scene_v4_{job_id}"\n    job_dir.mkdir(parents=True, exist_ok=True)\n    raw_cand: Optional[Path] = None\n    t0 = time.time()\n    try:\n        with JOBS_LOCK:\n            if job_id in JOBS:\n                JOBS[job_id]["status"] = "processing"\n                JOBS[job_id]["started_at"] = t0\n\n        img_p = job_dir / "keyframe.jpg"\n        img_p.write_bytes(base64.b64decode(req.image_b64))\n        wav_p: Optional[Path] = None\n        dur_target = float(req.duration_sec or 3.2)\n        if req.audio_b64:\n            wav_p = job_dir / "dialogue.wav"\n            wav_p.write_bytes(base64.b64decode(req.audio_b64))\n            dur_target = max(1.0, min(12.0, _probe_dur(wav_p)))\n\n        fps = int(req.fps or 24)\n        final_mp4 = job_dir / "final_scene.mp4"\n        resolved_model = _resolve_model_type(req.backend, has_audio=(wav_p is not None))\n\n        with GPU_LOCK:\n            session = _get_wangp_session()\n            meta = session.get_model_metadata(resolved_model)\n            if not meta:\n                avail = [\n                    (m.get("id") or m.get("model_type") or str(m))\n                    for m in (session.list_model_metadata() or [])[:25]\n                ]\n                raise RuntimeError(\n                    f"Modelo \'{resolved_model}\' não encontrado no WanGP instalado. Disponíveis (amostra): {avail}"\n                )\n\n            base_settings = session.get_default_settings(resolved_model)\n            mdef = session.get_model_def(resolved_model) or {}\n            schema = (session.get_model_schema(resolved_model) or {}).get("metadata", {})\n            settings = _build_wangp_task_settings(\n                base_settings,\n                resolved_model,\n                mdef,\n                schema,\n                prompt=req.prompt,\n                negative_prompt=req.negative_prompt,\n                img_path=img_p,\n                wav_path=wav_p,\n                dur_target=dur_target,\n                width=int(req.width),\n                height=int(req.height),\n                fps=fps,\n                steps=int(req.steps),\n                guidance_scale=float(req.guidance_scale),\n                seed=int(req.seed),\n            )\n\n            job = session.submit_task(settings)\n            result = job.result()\n            if not result.success or not result.generated_files:\n                errs = "; ".join(str(e) for e in (result.errors or [])) or "WanGP retornou 0 arquivos."\n                raise RuntimeError(errs[:600])\n\n            _LAST_LOADED_MODEL = resolved_model\n            raw_cand = Path(result.generated_files[0])\n            _mux_no_freeze(raw_cand, wav_p, final_mp4, int(req.width), int(req.height), fps=fps)\n            vqg = _validate_video_output(final_mp4, min_dur=0.3)\n            peak_vram = _smi_used_gb()\n            _cleanup_vram()\n\n        vid_b64 = base64.b64encode(final_mp4.read_bytes()).decode("ascii")\n        with JOBS_LOCK:\n            if job_id in JOBS:\n                JOBS[job_id].update({\n                    "status": "completed",\n                    "ok": True,\n                    "backend": resolved_model,\n                    "elapsed_s": round(time.time() - t0, 2),\n                    "duration": vqg["duration"],\n                    "vram_peak_gb": peak_vram,\n                    "video_b64": vid_b64,\n                })\n    except Exception as exc:\n        with JOBS_LOCK:\n            if job_id in JOBS:\n                JOBS[job_id].update({\n                    "status": "error",\n                    "ok": False,\n                    "elapsed_s": round(time.time() - t0, 2),\n                    "error": str(exc)[:600],\n                })\n    finally:\n        if raw_cand is not None:\n            try:\n                raw_cand.unlink(missing_ok=True)\n            except Exception:\n                pass\n        shutil.rmtree(job_dir, ignore_errors=True)\n        _cleanup_vram()\n\n\n@app.post("/jobs/animate_scene")\ndef submit_scene_job(req: AnimateSceneReq, x_colab_token: str = Header(default="")):\n    _require_token(x_colab_token, require_configured=False)\n    job_id = uuid.uuid4().hex[:10]\n    with JOBS_LOCK:\n        _evict_finished_jobs_locked(max_finished=16)\n        JOBS[job_id] = {"ok": True, "job_id": job_id, "status": "queued", "created_at": time.time()}\n    threading.Thread(target=_execute_scene_job, args=(job_id, req), daemon=True).start()\n    return {"ok": True, "job_id": job_id, "status": "queued"}\n\n\n@app.get("/jobs/{job_id}")\ndef get_scene_job(job_id: str, x_colab_token: str = Header(default="")):\n    _require_token(x_colab_token, require_configured=False)\n    with JOBS_LOCK:\n        job = JOBS.get(job_id)\n        if not job:\n            raise HTTPException(status_code=404, detail="Job não encontrado")\n        return dict(job)\n\n\nif __name__ == "__main__":\n    log_cfg = {\n        "version": 1,\n        "disable_existing_loggers": False,\n        "formatters": {\n            "default": {"format": "%(asctime)s [%(levelname)s] %(message)s"},\n        },\n        "handlers": {\n            "file": {\n                "class": "logging.FileHandler",\n                "filename": "/tmp/colab_v4_server.log",\n                "formatter": "default",\n            },\n            "stdout": {\n                "class": "logging.StreamHandler",\n                "stream": "ext://sys.stdout",\n                "formatter": "default",\n            },\n        },\n        "loggers": {\n            "uvicorn": {"handlers": ["file", "stdout"], "level": "WARNING", "propagate": False},\n            "uvicorn.error": {"handlers": ["file", "stdout"], "level": "WARNING", "propagate": False},\n            "uvicorn.access": {"handlers": ["file"], "level": "INFO", "propagate": False},\n        },\n    }\n    uvicorn.run(app, host="127.0.0.1", port=int(os.environ.get("PORT", "8787")), log_config=log_cfg)\n'

srv_file = WS / "colab_v4_cinema_server.py"
srv_file.write_text(SERVER_CODE.lstrip("\n"), encoding="utf-8")

# Bracket trick evita self-kill do subshell bash -lc
subprocess.run(
    ["bash", "-lc", "pkill -f '[c]olab_v4_cinema_server' || true; pkill -f '[c]loudflared' || true"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(1)

WANGP_VENV = WS / "envs/wangp"
PY_BIN = str(WANGP_VENV / "bin/python")
AUTH_TOKEN = env_cfg.get("COLAB_AUTH_TOKEN") or ""
HDRS = {"X-Colab-Token": AUTH_TOKEN} if AUTH_TOKEN else {}

extra_srv = {
    "PORT": "8787",
    "AI_STUDIO_ROOT": str(WS),
    "AI_STUDIO_VIDEO_MODEL": env_cfg["VIDEO_MODEL"],
    "COLAB_AUTH_TOKEN": AUTH_TOKEN,
}
if env_cfg.get("HF_TOKEN"):
    extra_srv["HF_TOKEN"] = env_cfg["HF_TOKEN"]

SRV_ENV = build_clean_env(ws=WS, venv_dir=WANGP_VENV, extra=extra_srv)
CF_ENV = build_clean_env(ws=WS)

def probe_health(url: str, timeout: float = 3.0):
    req = urllib.request.Request(url, headers=HDRS)
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read().decode("utf-8"))

def start_server():
    log_fp = open("/tmp/colab_v4_server.log", "a", encoding="utf-8")
    proc = subprocess.Popen(
        [PY_BIN, str(srv_file)],
        env=SRV_ENV,
        stdout=log_fp,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    health_data = None
    for _ in range(50):
        if proc.poll() is not None:
            log_fp.flush()
            tail = Path("/tmp/colab_v4_server.log").read_text(encoding="utf-8", errors="ignore")[-2500:]
            raise RuntimeError(f"Servidor FastAPI encerrou prematuramente (rc={proc.returncode}):\n{tail}")
        try:
            h = probe_health("http://127.0.0.1:8787/health", timeout=2.0)
            if h.get("ok"):
                health_data = h
                break
        except Exception:
            time.sleep(0.5)
    if not health_data:
        tail = Path("/tmp/colab_v4_server.log").read_text(encoding="utf-8", errors="ignore")[-1500:]
        raise RuntimeError(f"Timeout aguardando http://127.0.0.1:8787/health:\n{tail}")
    if not str(health_data.get("version", "")).startswith("4.0"):
        raise RuntimeError(f"Versão incompatível com vps_novela_runner.py: {health_data.get('version')}")
    return proc, log_fp, health_data

def start_tunnel():
    cf_log_path = Path("/tmp/cloudflared_v4.log")
    cf_log_path.write_text("", encoding="utf-8")
    log_fp = open(cf_log_path, "a", encoding="utf-8")
    proc = subprocess.Popen(
        [
            "cloudflared", "tunnel",
            "--no-autoupdate",
            "--protocol", "http2",
            "--url", "http://127.0.0.1:8787",
            "--logfile", str(cf_log_path),
            "--loglevel", "info",
        ],
        env=CF_ENV,
        stdout=log_fp,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    tunnel_url = ""
    for _ in range(60):
        if proc.poll() is not None:
            tail = cf_log_path.read_text(encoding="utf-8", errors="ignore")[-1200:]
            raise RuntimeError(f"cloudflared encerrou prematuramente (rc={proc.returncode}):\n{tail}")
        txt = cf_log_path.read_text(encoding="utf-8", errors="ignore")
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", txt)
        if m:
            tunnel_url = m.group(0)
            break
        time.sleep(0.5)
    if not tunnel_url:
        tail = cf_log_path.read_text(encoding="utf-8", errors="ignore")[-1200:]
        raise RuntimeError(f"Não foi possível obter URL trycloudflare.com em 30s:\n{tail}")

    public_ok = False
    for _ in range(15):
        try:
            h = probe_health(f"{tunnel_url}/health", timeout=4.0)
            if h.get("ok"):
                public_ok = True
                break
        except Exception:
            time.sleep(1.0)
    return proc, log_fp, tunnel_url, public_ok

def print_banner(url: str, public_ok: bool, h: dict):
    status_txt = "ONLINE (probe externo OK)" if public_ok else "ATIVO (aguardando propagação DNS)"
    print("\n====================================================================")
    print(f"PONTE L4 {h.get('version')} {status_txt}")
    print(f"URL: {url}")
    print(
        f"GPU={h.get('gpu')} | VRAM={h.get('vram_free_gb')}/{h.get('vram_gb')}GB | "
        f"Model={h.get('default_video_model')} | SmokeOK={h.get('smoke_verified')} | "
        f"EnvSanitized={h.get('env_sanitized')}"
    )
    print("====================================================================\n", flush=True)

Path("/tmp/colab_v4_server.log").write_text("", encoding="utf-8")
srv_proc, srv_log, health_data = start_server()
print(
    f"[OK] Worker FastAPI {health_data.get('version')} ativo | GPU={health_data.get('gpu')} | "
    f"VRAM={health_data.get('vram_free_gb')}/{health_data.get('vram_gb')}GB | "
    f"model={health_data.get('default_video_model')} | ready={health_data.get('ready')}"
)

cf_proc, cf_log, tunnel_url, pub_ok = start_tunnel()
print_banner(tunnel_url, pub_ok, health_data)

srv_restarts = 0
last_hb = time.time()
try:
    while True:
        time.sleep(15)
        if srv_proc.poll() is not None:
            srv_restarts += 1
            tail = Path("/tmp/colab_v4_server.log").read_text(encoding="utf-8", errors="ignore")[-1500:]
            print(f"[WARN] Servidor encerrou (rc={srv_proc.returncode}, tentativa {srv_restarts}/3). Últimas linhas:\n{tail}", flush=True)
            if srv_restarts > 3:
                raise RuntimeError("Servidor FastAPI falhou mais de 3 vezes consecutivas.")
            time.sleep(3)
            srv_proc, srv_log, health_data = start_server()
            print("[OK] Servidor FastAPI recuperado automaticamente.", flush=True)

        if cf_proc.poll() is not None:
            print(f"[WARN] Túnel cloudflared caiu (rc={cf_proc.returncode}) — reiniciando túnel...", flush=True)
            time.sleep(2)
            cf_proc, cf_log, tunnel_url, pub_ok = start_tunnel()
            print_banner(tunnel_url, pub_ok, health_data)

        if time.time() - last_hb >= 60:
            last_hb = time.time()
            try:
                h = probe_health("http://127.0.0.1:8787/health", timeout=3.0)
                print(
                    f"[HB {time.strftime('%H:%M:%S')}] VRAM Livre: {h.get('vram_free_gb')}/{h.get('vram_gb')}GB | "
                    f"Disco Livre: {h.get('cache_free_gb')}GB | Jobs: {h.get('active_jobs')} | URL: {tunnel_url}",
                    flush=True,
                )
            except Exception as exc:
                print(f"[WARN {time.strftime('%H:%M:%S')}] Falha no heartbeat local: {exc}", flush=True)
finally:
    for p in (cf_proc, srv_proc):
        try:
            p.terminate()
        except Exception:
            pass
    for fp in (cf_log, srv_log):
        try:
            fp.close()
        except Exception:
            pass
